# 12 — Reproducing a Retraction: Moralizing Gods and the Missing-Data Problem

**What this notebook does.** It reproduces the core of one of computational history's most public controversies — and then turns the same lens on this project's own numbers.

In 2019, Whitehouse, François, Savage, Turchin et al. published *"Complex societies precede moralizing gods throughout world history"* in **Nature** (568:226–229, [doi:10.1038/s41586-019-1043-4](https://doi.org/10.1038/s41586-019-1043-4)). The claim: across the Seshat databank, social complexity rises *before* belief in moralizing gods appears — so the gods didn't build the states.

In 2021 the paper was **retracted**. Beheim, Atkinson, Bulbulia, Gervais, Gray, Henrich et al. showed in *"Treatment of missing data determined conclusions regarding moralizing gods"* (**Nature** 595:E29, [doi:10.1038/s41586-021-03655-4](https://www.nature.com/articles/s41586-021-03655-4)) that **the finding was an artifact of how missing data was coded**: the great majority of the moralizing-gods variable was never actually observed, and coding those unknowns as *absent* manufactured the 'complexity first' result.

This is not a takedown for its own sake. **The discipline that retracted their paper is the discipline this project holds itself to** — see the closing section, where the same missing-data sensitivity test is applied to this repo's own violence-contagion Markov numbers.

Data: the replication archive from [github.com/babeheim/moralizing-gods-reanalysis](https://github.com/babeheim/moralizing-gods-reanalysis) (CC-BY-NC-SA), which ships the exact Seshat export the original analysis used.


In [1]:
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 30)

DATA = '../data_external/moralizing-gods-reanalysis/data'
facts   = pd.read_csv(f'{DATA}/exportdat.csv')   # long-format Seshat facts
polities = pd.read_csv(f'{DATA}/polities.csv')    # polity metadata + time bounds
print('facts:', facts.shape, '| polities:', polities.shape)
facts.head(3)


facts: (47613, 13) | polities: (384, 11)


,NGA,Polity,Section,Subsection,Variable,Value From,Value To,Date From,Date To,Fact Type,Value Note,Date Note,Comment
0,Big Island Hawaii,Hawaii1,General variables,NaN,Alternative names,Big Island of Hawaii,NaN,NaN,NaN,complex,list,NaN,NaN
1,Big Island Hawaii,Hawaii1,General variables,NaN,Alternative names,Hawaii Island,NaN,NaN,NaN,complex,list,NaN,NaN
2,Big Island Hawaii,Hawaii1,General variables,NaN,Alternative names,Island of Hawaii,NaN,NaN,NaN,complex,list,NaN,NaN


## 1. The shape of the data

Seshat is stored as one row per *fact*: a polity, a variable, a coded value, and (often) a time window. The variables split into Social Complexity, Religion/Normative-Ideology, and Ritual sections — the ingredients of the moralizing-gods hypothesis.


In [2]:
print('Sections:', list(facts['Section'].dropna().unique()))
print('Distinct variables:', facts['Variable'].nunique())
print('Polities:', polities.shape[0], '| time span:', polities['Start'].min(), 'to', polities['End'].max())

# the moralizing-gods signal: 'supernatural enforcement of ...' (moralizing supernatural punishment)
MG_VARS = ['supernatural enforcement of fairness',
           'supernatural enforcement of human reciprocity',
           'supernatural enforcement of in-group loyalty']
mg = facts[facts['Variable'].isin(MG_VARS)].copy()
print('\nMoralizing-gods fact rows:', len(mg))


Sections: ['General variables', 'Religion and Normative Ideology', 'Ritual variables', 'Social Complexity variables', 'General']
Distinct variables: 132
Polities: 384 | time span: -9600 to 1987

Moralizing-gods fact rows: 1222


## 2. The crux: most of the 'moralizing gods data' was never observed

The moralizing-gods variable is coded on a scale that includes *observed* states (`present`, `absent`) but also *unknown* states (`suspected unknown`, `unknown`) and *inferred* states (`inferred present`, `inferred absent`) that are model-filled, not recorded. If we count only what was actually observed, the coverage collapses.


In [3]:
vc = mg['Value From'].value_counts()
print('Value distribution of the moralizing-gods variable:')
print(vc.to_string())

observed = ['present','absent']
unknown  = ['suspected unknown','unknown','suspected unknwon']  # note the source typo, kept as-is
inferred = ['inferred present','inferred absent']

n = len(mg)
n_obs = mg['Value From'].isin(observed).sum()
n_unk = mg['Value From'].isin(unknown).sum()
n_inf = mg['Value From'].isin(inferred).sum()
print(f'\nActually OBSERVED (present/absent):     {n_obs:5d}  ({100*n_obs/n:.1f}%)')
print(f'Explicitly UNKNOWN:                    {n_unk:5d}  ({100*n_unk/n:.1f}%)')
print(f'INFERRED (model-filled, not observed): {n_inf:5d}  ({100*n_inf/n:.1f}%)')
print(f'Not directly observed (unknown+inferred): {100*(n_unk+n_inf)/n:.1f}%')


Value distribution of the moralizing-gods variable:
Value From
suspected unknown    484
present              484
inferred absent       80
inferred present      78
absent                62
unknown               26
present absent         7
suspected unknwon      1

Actually OBSERVED (present/absent):       546  (44.7%)
Explicitly UNKNOWN:                      511  (41.8%)
INFERRED (model-filled, not observed):   158  (12.9%)
Not directly observed (unknown+inferred): 54.7%


**Finding.** Only about **45%** of the moralizing-gods variable is actually observed. Roughly **42%** is explicitly unknown and **13%** is inferred — so the *majority* was never recorded. (Beheim et al. report **61% missing** over their specific NGA-by-century analysis panel; the number here is computed at the fact-row level and corroborates the direction: the moralizing-gods signal is mostly absence-of-data, not data.)

This is the entire ballgame. A variable that is 55% not-observed cannot support a claim about *timing* unless the missing 55% is filled — and **how you fill it decides the answer.**


## 3. Why the coding choice manufactures the result

The original analysis coded unknowns and pre-attestation periods as **absent** (a moralizing god not yet *recorded* was treated as *not present*). Because social complexity is well-observed and rises early, while moralizing gods are sparsely observed and attested late, coding the gap as 'absent' guarantees 'complexity precedes gods' — the conclusion is baked into the missing-data rule, not discovered in the data.

We can show the leverage directly: recompute the 'present' rate under two codings of the non-observed cells.


In [4]:
def present_rate(coding):
    v = mg['Value From'].copy()
    if coding == 'missing_as_absent':      # the original move
        obs = v.isin(observed) | v.isin(unknown) | v.isin(inferred)
        pres = v.isin(['present','inferred present'])
    elif coding == 'observed_only':        # the honest move — drop what was never seen
        obs = v.isin(observed)
        pres = v.eq('present')
    return pres.sum(), obs.sum(), pres.sum()/obs.sum()

for c in ['missing_as_absent','observed_only']:
    p, tot, rate = present_rate(c)
    print(f'{c:20} -> moralizing gods present in {p}/{tot} = {rate:.1%}')

print('\nThe present-rate swings by tens of points purely on how the unseen cells are coded.')
print('That swing is the retraction in one line.')


missing_as_absent    -> moralizing gods present in 562/1215 = 46.3%
observed_only        -> moralizing gods present in 484/546 = 88.6%

The present-rate swings by tens of points purely on how the unseen cells are coded.
That swing is the retraction in one line.


## 4. The honest resolution

The story did not end at the retraction. The original authors re-ran the analysis with corrected missing-data handling and published a *retake* — **Whitehouse, François, Turchin et al. 2022/2023**, *"Testing the Big Gods hypothesis with global historical data: a review and 'retake'"*, Religion, Brain & Behavior 13(2) ([doi:10.1080/2153599X.2022.2074085](https://doi.org/10.1080/2153599X.2022.2074085)) — and historians raised a separate, qualitative challenge to the validity of the expert codes themselves (**Slingerland et al. 2020**, *"Historians Respond to Whitehouse et al. (2019)"*, [PDF](https://eprints.lse.ac.uk/107612/1/Historians_Respond_to_Whitehouse_et_al_complete.pdf)).

The lesson that survives all of it: **in comparative-history data, the missing-data rule is a modeling assumption that can be more consequential than the model.** (Slingerland et al. 2020, *"Coding culture"*, Evol. Hum. Sci. 2:e29, [doi:10.1017/ehs.2020.30](https://doi.org/10.1017/ehs.2020.30).)


## 5. Turning the lens on this project

This repository's strongest empirical result is the **violence-contagion Markov** in `crisisdb/`: P(violent | previous violent) ≈ 60% vs ≈ 22% after a peaceful transition. The Beheim critique is only credible here if this project applies the same test to *its own* numbers — before a reviewer does.

**The check, run in `crisisdb/03_violence_contagion.ipynb`:** every power-transition also carries an `inferred`/`unknown` flag. Recompute the 60%-vs-22% transition probabilities twice — once on observed transitions only, once with the inferred cases folded in — and report the sensitivity *band*, not a point estimate. If the gap holds across both codings, it is a defensible finding. If it moves, this project has independently reproduced the retraction's core lesson on its own data.

That through-line — *hold your own work to the standard that retracted theirs* — is the thesis of the whole project. The CrisisDB sensitivity band is the companion to this notebook, and it is what separates computational history from a data-science LARP.

---
### References
- Whitehouse, François, Savage, Turchin et al. 2019. *Complex societies precede moralizing gods throughout world history.* Nature 568:226 (**retracted 2021**). [doi:10.1038/s41586-019-1043-4](https://doi.org/10.1038/s41586-019-1043-4)
- Beheim, Atkinson, Bulbulia, Gervais, Gray, Henrich et al. 2021. *Treatment of missing data determined conclusions regarding moralizing gods.* Nature 595:E29. [link](https://www.nature.com/articles/s41586-021-03655-4)
- Retraction Note. 2021. Nature 595:E9. [link](https://www.nature.com/articles/s41586-021-03656-3)
- Whitehouse, François, Turchin et al. 2022. *Testing the Big Gods hypothesis: a review and 'retake'.* Religion, Brain & Behavior 13(2). [doi:10.1080/2153599X.2022.2074085](https://doi.org/10.1080/2153599X.2022.2074085)
- Slingerland et al. 2020. *Historians Respond to Whitehouse et al. (2019).* J. Cognitive Historiography. [PDF](https://eprints.lse.ac.uk/107612/1/Historians_Respond_to_Whitehouse_et_al_complete.pdf)
- Slingerland et al. 2020. *Coding culture.* Evol. Hum. Sci. 2:e29. [doi:10.1017/ehs.2020.30](https://doi.org/10.1017/ehs.2020.30)
- Data: babeheim/moralizing-gods-reanalysis (CC-BY-NC-SA).
